# EX_00 — PyTorch vs TensorFlow (ejercicios)

**Notebook de referencia:** `notebook/00_Frameworks_Pytorch_vs_Tensorflow.ipynb`

**Tiempo orientativo:** ~30 minutos.

En esta hoja practicarás ideas equivalentes en ambos frameworks: tensores, capas lineales y un forward pass mínimo.


## Actividad 1 — Activación a mano

Implementa en NumPy una función `relu` y otra `sigmoid` y comprueba que coinciden con `torch` y `tensorflow` en un vector de prueba.

*Hint:* use `torch.relu`, `tf.nn.relu`; for sigmoid use `torch.sigmoid` and `tf.nn.sigmoid`.


In [1]:
import numpy as np
import torch
import tensorflow as tf

def relu_np(z): return np.maximum(0, z)
def sigmoid_np(z): return 1 / (1 + np.exp(-z))

x = np.array([-2.0, 0.0, 1.5], dtype=np.float32)

# Assertions para comprobar que coinciden
assert np.allclose(relu_np(x), torch.relu(torch.tensor(x)).numpy())
assert np.allclose(relu_np(x), tf.nn.relu(tf.constant(x)).numpy())
assert np.allclose(sigmoid_np(x), torch.sigmoid(torch.tensor(x)).numpy())
print("¡Todo coincide perfectamente!")

2026-05-25 18:03:34.353752: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


¡Todo coincide perfectamente!


## Actividad 2 — Misma arquitectura, dos APIs

Define una red `Linear(10, 3)` + `ReLU` + `Linear(3, 1)` en **PyTorch** (`nn.Sequential`) y la misma en **Keras** (`Sequential`).

*Hint:* set seeds (`torch.manual_seed`, `tf.random.set_seed`) and use explicit init if you want to compare weights.


In [2]:
import torch
import torch.nn as nn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# PyTorch
torch.manual_seed(42)
torch_model = nn.Sequential(
    nn.Linear(10, 3),
    nn.ReLU(),
    nn.Linear(3, 1)
)

# Keras
tf.random.set_seed(42)
keras_model = keras.Sequential([
    layers.Input(shape=(10,)),
    layers.Dense(3, activation='relu'),
    layers.Dense(1)
])

# Forward pass
t_out = torch_model(torch.randn(4, 10))
k_out = keras_model(tf.random.normal((4, 10)))
print("PyTorch output shape:", t_out.shape)
print("Keras output shape:", k_out.shape)

PyTorch output shape: torch.Size([4, 1])
Keras output shape: (4, 1)


## Actividad 3 — Entrenamiento en mini-batch (conceptual + código corto)

Escribe un bucle de **una época** que: (1) muestree un batch sintético `X, y` para regresión, (2) calcule `MSE`, (3) haga `backward` / `gradient` y un paso de optimizador.

Elige **solo uno** de los dos frameworks para el bucle completo; en el otro, documenta en un comentario qué API usarías (`loss.backward`, `tape.gradient`, etc.).


In [3]:
# Bucle en PyTorch (Comentando la API equivalente de TF)
"""
# En TensorFlow usaríamos:
with tf.GradientTape() as tape:
    pred = model(X)
    loss = mse_loss(y, pred)
grads = tape.gradient(loss, model.trainable_variables)
optimizer.apply_gradients(zip(grads, model.trainable_variables))
"""

# Implementación PyTorch conceptual
optimizer = torch.optim.SGD(torch_model.parameters(), lr=0.01)
criterion = nn.MSELoss()

X = torch.randn(8, 10) # Mini-batch sintético
y = torch.randn(8, 1)

optimizer.zero_grad()  # 1. Limpiar gradientes
preds = torch_model(X) # 2. Forward
loss = criterion(preds, y) # 3. Loss (MSE)
loss.backward()        # 4. Backward
optimizer.step()       # 5. Paso del optimizador
print(f"Loss epoch 1: {loss.item():.4f}")

Loss epoch 1: 0.4466
